# SightSeeker: fine-tune YOLO26 on a free Colab GPU
1. **Runtime → Change runtime type → T4 GPU**.
2. Zip your YOLO-format dataset (folder with `data.yaml`, `images/`, `labels/`) as `dataset.zip`.
3. Run the cells top to bottom. At the end, `best.pt` downloads; put it in your SightSeeker folder and run
   `python yolo_detection.py --model best.pt`.

No dataset yet? Run the smoke-test cell (`coco8`) to check everything works first.

In [ ]:
!pip -q install ultralytics
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

In [ ]:
# Optional smoke test on 8 images (downloads automatically from GitHub)
from ultralytics import YOLO
YOLO('yolo26s.pt').train(data='coco8.yaml', epochs=3, imgsz=640)

In [ ]:
# Upload dataset.zip (YOLO format)
from google.colab import files
up = files.upload()
!unzip -q -o dataset.zip -d dataset
!find dataset -name data.yaml

In [ ]:
# Train. Edit DATA if the data.yaml path printed above is different.
DATA = 'dataset/data.yaml'
from ultralytics import YOLO
model = YOLO('yolo26s.pt')
model.train(data=DATA, epochs=60, imgsz=640, batch=16, freeze=10, patience=15, name='sightseeker', exist_ok=True)
m = model.val(data=DATA)
print('mAP50-95:', round(m.box.map, 3), ' mAP50:', round(m.box.map50, 3))

In [ ]:
# Download the trained weights
from google.colab import files
files.download('runs/detect/sightseeker/weights/best.pt')